# TrashScan — Path A com múltiplos modelos YOLO

Este notebook foi preparado para rodar **localmente no VS Code/Jupyter**, assumindo que você vai:

1. fazer `git clone` do repositório `TrashScan`
2. abrir este notebook **de dentro do clone**
3. baixar **TACO** e **Roboflow** fora do repositório, como no fluxo anterior
4. usar o **Path A** para treinar **vários modelos YOLO** e comparar os resultados

## O que este notebook cobre

- descoberta da raiz do repositório
- configuração de caminhos locais
- merge e preprocessamento com `--path A`
- treino em grade de múltiplos modelos YOLO
- avaliação individual, em lote e com TTA/WBF
- geração de resumo global dos benchmarks

## Arquivos-base considerados

- `train_path_A.py` enviado por você, que treina uma grade de modelos e salva métricas por modelo
- `evaluate_tta.py` enviado por você, que faz avaliação unificada, suporta `--runs_dir`, TTA/WBF e resumo global

> Observação: este notebook assume a interface desses scripts como fonte da verdade.


## 1) Descoberta automática da raiz do repositório

A célula abaixo tenta localizar a raiz do clone do `TrashScan` mesmo se o notebook estiver dentro de `notebooks/`.


In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)

REPO_ROOT = None
for p in candidates:
    if (p / "data").exists() and (p / "train").exists() and (p / "eval").exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOK_DIR = cwd
PARENT_DIR = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
TRAIN_DIR = REPO_ROOT / "train" / "paths"
EVAL_DIR = REPO_ROOT / "eval"
CONFIG_DIR = REPO_ROOT / "configs"
UTILS_DIR = REPO_ROOT / "utils"

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("REPO_ROOT    =", REPO_ROOT)
print("PARENT_DIR   =", PARENT_DIR)


## 2) Configuração de caminhos

Por padrão, esta versão coloca datasets e artefatos **fora do repo**, no diretório pai do clone.  
Isso evita poluir o repositório e facilita reaproveitar dados entre execuções.

Você pode ajustar essas pastas se quiser.


In [ ]:
EXTERNAL_DIR = PARENT_DIR / "external_datasets"
TACO_DIR = PARENT_DIR / "TACO"
PROCESSED_DIR = PARENT_DIR / "processed_4cls"
RUNS_PATH_A_DIR = PARENT_DIR / "runs" / "path_A"
RESULTS_PATH_A_DIR = PARENT_DIR / "results_path_A"

for p in [EXTERNAL_DIR, TACO_DIR, PROCESSED_DIR, RUNS_PATH_A_DIR, RESULTS_PATH_A_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_YAML_PATH_A = PROCESSED_DIR / "dataset_path_A.yaml"

print("EXTERNAL_DIR       =", EXTERNAL_DIR)
print("TACO_DIR           =", TACO_DIR)
print("PROCESSED_DIR      =", PROCESSED_DIR)
print("RUNS_PATH_A_DIR    =", RUNS_PATH_A_DIR)
print("RESULTS_PATH_A_DIR =", RESULTS_PATH_A_DIR)
print("DATASET_YAML_PATH_A=", DATASET_YAML_PATH_A)


## 3) Verificação dos scripts usados

Aqui nós garantimos que os principais scripts do pipeline existem no clone local.


In [ ]:
required_paths = [
    DATA_DIR / "merge_datasets.py",
    DATA_DIR / "preprocess.py",
    TRAIN_DIR / "train_path_A.py",
    EVAL_DIR / "evaluate.py",
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos ausentes:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Há scripts ausentes no clone. Veja a lista acima.")
else:
    print("Todos os scripts principais foram encontrados.")


## 4) Ambiente Python

Você pode usar o ambiente do projeto (`conda`) ou instalar pelo notebook.

### Opção recomendada
No terminal:

```bash
conda env create -f env/environment_AB.yml
conda activate <nome-do-ambiente>
```

A célula abaixo mostra qual Python/Jupyter está ativo.


In [ ]:
import sys
print(sys.executable)


### Instalação opcional via notebook

Descomente só se você realmente precisar instalar dependências daqui.


In [ ]:
# !{sys.executable} -m pip install -U pip
# !{sys.executable} -m pip install ultralytics mlflow pandas pyyaml matplotlib scikit-learn tqdm opencv-python pycocotools roboflow ensemble-boxes


## 5) Utilitários de execução

As próximas células usam `subprocess` para rodar scripts do repositório de forma previsível no VS Code/Jupyter.


In [ ]:
import subprocess
import shlex
import os
from pathlib import Path

def run_cmd(cmd, cwd=PARENT_DIR, env=None):
    if isinstance(cmd, str):
        print("$", cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


## 6) GPU / CPU

O treino de vários YOLOs pode ficar pesado. Esta célula detecta CUDA e sugere um `BATCH` inicial.


In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = "0"
    if "A100" in gpu_name:
        BATCH = 64
    elif "V100" in gpu_name:
        BATCH = 32
    else:
        BATCH = 16
else:
    gpu_name = "cpu"
    DEVICE = "cpu"
    BATCH = 8

EPOCHS = 100
IMGSZ = 640
PATIENCE = 30

print("Dispositivo:", gpu_name)
print("DEVICE     :", DEVICE)
print("BATCH      :", BATCH)
print("EPOCHS     :", EPOCHS)
print("IMGSZ      :", IMGSZ)
print("PATIENCE   :", PATIENCE)


## 7) Download do TACO

Como você comentou que o TACO continuará vindo de fora do repositório, esta etapa fica separada.

Se o seu script `data/download_external_datasets.py` já cobre isso, veja a ajuda primeiro.


In [ ]:
downloader = DATA_DIR / "download_external_datasets.py"
if downloader.exists():
    run_cmd([sys.executable, str(downloader), "--help"])
else:
    print("Script de download externo não encontrado.")


### Verificação manual do TACO

Se você baixou o TACO manualmente, rode a célula abaixo antes de seguir.


In [ ]:
print("TACO_DIR existe?", TACO_DIR.exists())
if TACO_DIR.exists():
    for p in list(TACO_DIR.iterdir())[:10]:
        print(" -", p.name)


## 8) Download do dataset extra via Roboflow

Nesta versão, a chave da API fica em variável de ambiente.

No terminal Linux/macOS:

```bash
export ROBOFLOW_API_KEY="SUA_CHAVE"
```

No PowerShell:

```powershell
$env:ROBOFLOW_API_KEY="SUA_CHAVE"
```


In [ ]:
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "")
print("ROBOFLOW_API_KEY carregada?", bool(ROBOFLOW_API_KEY))


### Download opcional via Roboflow

Descomente e ajuste os identificadores do workspace/projeto/versão conforme o seu caso real.


In [ ]:
# from roboflow import Roboflow
#
# rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# project = rf.workspace("sadis-workspace").project("taco-dataset-ql1ng-atu1k")
# version = project.version(3)
# dataset = version.download("coco")
# print("Dataset baixado em:", dataset.location)


## 9) Organização do dataset externo

No fluxo anterior, os datasets externos eram reunidos em uma pasta que depois entrava no merge.  
A célula abaixo cria o destino padrão.


In [ ]:
COCO_EXTERNAL_DIR = EXTERNAL_DIR / "coco_format"
COCO_EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
print("Destino esperado para datasets externos:", COCO_EXTERNAL_DIR)


## 10) Merge dos datasets

Este passo combina TACO e os datasets externos usando o script do projeto.


In [ ]:
merge_script = DATA_DIR / "merge_datasets.py"

run_cmd([
    sys.executable, str(merge_script),
    "--taco_root", str(TACO_DIR),
    "--external_root", str(EXTERNAL_DIR),
    "--output_root", str(PROCESSED_DIR),
    "--skip_preprocess",
])


## 11) Pré-processamento para o Path A

Aqui está a troca principal em relação ao notebook anterior: agora o preprocessamento roda com `--path A`.


In [ ]:
preprocess_script = DATA_DIR / "preprocess.py"

run_cmd([
    sys.executable, str(preprocess_script),
    "--taco_root", str(PROCESSED_DIR / "merged_data"),
    "--output_root", str(PROCESSED_DIR),
    "--path", "A",
])


## 12) Validação dos artefatos processados

Esta etapa ajuda a detectar cedo problemas de estrutura do dataset.


In [ ]:
validate_script = UTILS_DIR / "validate_all.py"

if validate_script.exists():
    run_cmd([
        sys.executable, str(validate_script),
        "--check_processed",
        "--processed_dir", str(PROCESSED_DIR),
    ])
else:
    print("validate_all.py não encontrado; seguindo sem essa validação.")


## 13) Conferência do YAML do Path A

O script `train_path_A.py` espera um `--data` apontando para o YAML do dataset processado.


In [ ]:
print("DATASET_YAML_PATH_A =", DATASET_YAML_PATH_A)
print("Existe?", DATASET_YAML_PATH_A.exists())

if DATASET_YAML_PATH_A.exists():
    print(DATASET_YAML_PATH_A.read_text()[:1500])
else:
    raise FileNotFoundError(
        f"Dataset YAML do Path A não encontrado em {DATASET_YAML_PATH_A}. "
        "Confirme a saída do preprocess."
    )


## 14) Escolha dos modelos YOLO para o grid

O `train_path_A.py` enviado por você aceita uma grade de modelos via `--models`.

Exemplos suportados no script-base:
- `yolov8n`, `yolov8s`, `yolov8m`, `yolov8l`
- `yolov9s`, `yolov9c`, `yolov9e`
- `yolov10n`, `yolov10s`, `yolov10m`
- `yolov11n`, `yolov11s`, `yolov11m`, `yolov11l`, `yolov11x`
- `yolov5su`
- `rtdetr-l`, `rtdetr-x`

Ajuste a lista abaixo conforme sua GPU e tempo disponível.


In [ ]:
MODELS = [
    "yolov8n",
    "yolov8s",
    "yolov9s",
    "yolov11n",
]

print("Modelos selecionados:", MODELS)


## 15) Treino do Path A para múltiplos modelos

Esta célula roda o benchmark de treino usando o script `train/paths/train_path_A.py`.

Os resultados ficam em:
- `RUNS_PATH_A_DIR/<modelo>/...`
- `mlruns/` (MLflow), se configurado pelo script


In [ ]:
train_script = TRAIN_DIR / "train_path_A.py"

run_cmd([
    sys.executable, str(train_script),
    "--data", str(DATASET_YAML_PATH_A),
    "--output", str(RUNS_PATH_A_DIR),
    "--models", *MODELS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--imgsz", str(IMGSZ),
    "--device", str(DEVICE),
    "--patience", str(PATIENCE),
])


## 16) Resumo rápido dos treinos

O próprio `train_path_A.py` permite resumir os `metrics.json` já salvos.


In [ ]:
run_cmd([
    sys.executable, str(TRAIN_DIR / "train_path_A.py"),
    "--summarize",
    "--output", str(RUNS_PATH_A_DIR),
])


## 17) Estrutura dos pesos treinados

Verifique se os modelos produziram `weights/best.pt`.


In [ ]:
best_weights = sorted(RUNS_PATH_A_DIR.glob("*/weights/best.pt"))
print(f"Pesos encontrados: {len(best_weights)}")
for p in best_weights:
    print(" -", p)


## 18) Avaliação em lote do Path A

O avaliador unificado percorre todos os `best.pt` abaixo de `--runs_dir` e produz:
- arquivos individuais por modelo
- gráficos
- resumo global do benchmark


In [ ]:
evaluate_script = EVAL_DIR / "evaluate.py"

run_cmd([
    sys.executable, str(evaluate_script),
    "--path", "A",
    "--runs_dir", str(RUNS_PATH_A_DIR),
    "--data_yaml", str(DATASET_YAML_PATH_A),
    "--output", str(RESULTS_PATH_A_DIR),
    "--device", str(DEVICE),
    "--imgsz", str(IMGSZ),
])


## 19) Avaliação com TTA + WBF

Use esta célula para uma avaliação mais forte no inference-time.  
Ela costuma ser mais lenta, mas pode melhorar métricas.


In [ ]:
# Descomente para rodar TTA + WBF
# run_cmd([
#     sys.executable, str(EVAL_DIR / "evaluate.py"),
#     "--path", "A",
#     "--runs_dir", str(RUNS_PATH_A_DIR),
#     "--data_yaml", str(DATASET_YAML_PATH_A),
#     "--output", str(RESULTS_PATH_A_DIR),
#     "--device", str(DEVICE),
#     "--imgsz", str(IMGSZ),
#     "--use_tta_wbf",
#     "--tta_scales", "512", "640", "768",
#     "--tta_flip",
#     "--tta_wbf_iou", "0.55",
#     "--tta_skip_box_thr", "0.001",
# ])


## 20) Ensemble opcional entre modelos

O avaliador também aceita ensemble WBF entre todos os modelos encontrados em `--runs_dir`.
Use com cuidado: isso aumenta o custo de inferência.


In [ ]:
# Descomente para avaliar ensemble
# run_cmd([
#     sys.executable, str(EVAL_DIR / "evaluate.py"),
#     "--path", "A",
#     "--runs_dir", str(RUNS_PATH_A_DIR),
#     "--data_yaml", str(DATASET_YAML_PATH_A),
#     "--output", str(RESULTS_PATH_A_DIR),
#     "--device", str(DEVICE),
#     "--imgsz", str(IMGSZ),
#     "--ensemble",
#     "--ensemble_iou", "0.55",
# ])


## 21) Geração do resumo global

Esta célula recompila o ranking global a partir dos JSONs individuais salvos em `RESULTS_PATH_A_DIR`.


In [ ]:
run_cmd([
    sys.executable, str(EVAL_DIR / "evaluate.py"),
    "--summarize",
    "--output", str(RESULTS_PATH_A_DIR),
])


## 22) Inspeção rápida dos resultados

Aqui você consegue conferir os principais arquivos gerados.


In [ ]:
from pathlib import Path

for sub in [RESULTS_PATH_A_DIR / "individual", RESULTS_PATH_A_DIR / "global", RESULTS_PATH_A_DIR / "plots"]:
    print("\n==", sub, "==")
    if sub.exists():
        files = sorted(sub.glob("*"))
        for p in files[:20]:
            print(" -", p.name)
    else:
        print("Pasta não encontrada.")


## 23) Leitura do summary CSV em pandas

Se o `benchmark_summary.csv` existir, esta célula carrega a tabela para inspeção rápida.


In [ ]:
import pandas as pd

summary_csv = RESULTS_PATH_A_DIR / "global" / "benchmark_summary.csv"
if summary_csv.exists():
    df_summary = pd.read_csv(summary_csv)
    display(df_summary)
else:
    print("Arquivo não encontrado:", summary_csv)


## 24) Notas finais

### Ordem sugerida de execução
1. descobrir repositório e configurar caminhos
2. confirmar ambiente Python
3. garantir TACO + datasets externos
4. rodar merge
5. rodar preprocess com `--path A`
6. conferir `dataset_path_A.yaml`
7. escolher a lista `MODELS`
8. treinar
9. avaliar
10. gerar resumo global

### Ajustes que você pode querer fazer
- reduzir `MODELS` se sua GPU for limitada
- diminuir `EPOCHS`
- baixar `BATCH`
- rodar avaliação padrão primeiro e TTA depois
- usar ensemble só no fim
